# Assignment

### Leveraging NoSQL functionality in a RDBMS

In our last lecturer we have discovered that we can easily blend json and sql data using PostgreSQL's JSONB datatype and its operators.

Now it's your task to provide some answers using the migrated json data.


## Query jtracks data

We would like to evaluate the data and generate some reports.

### Most popular artists
_Which artists have been listened to most often and how often?_ 

In [24]:
# create docker container to run database
!docker-compose -f docker-compose.yml -p postgres up -d
!docker cp music-store_ALLTOGETHER.sql postgres-db-1:/music-store.sql
!docker exec -i postgres-db-1 bash -c "psql -U admin -d postgres < music-store.sql"

 Container postgres-db-1  Running


SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
CREATE EXTENSION
COMMENT
SET
SET
 setval 
--------
    203
(1 row)

 setval 
--------
    347
(1 row)

 setval 
--------
    275
(1 row)

 setval 
--------
     16
(1 row)

 setval 
--------
     59
(1 row)

 setval 
--------
      8
(1 row)

 setval 
--------
   1000
(1 row)

 setval 
--------
     25
(1 row)

 setval 
--------
   2240
(1 row)

 setval 
--------
    412
(1 row)

 setval 
--------
      5
(1 row)

 setval 
--------
     18
(1 row)

 setval 
--------
   3503
(1 row)

CREATE TABLE
INSERT 0 318


ERROR:  function "last_updated" already exists with same argument types
ERROR:  relation "actors_id_seq" already exists
ERROR:  relation "actors" already exists
ERROR:  relation "albums_id_seq" already exists
ERROR:  relation "albums" already exists
ERROR:  relation "artists_id_seq" already exists
ERROR:  relation "artists" already exists
ERROR:  relation "categories_id_seq" already exists
ERROR:  relation "categories" already exists
ERROR:  relation "customers_id_seq" already exists
ERROR:  relation "customers" already exists
ERROR:  relation "employees_id_seq" already exists
ERROR:  relation "employees" already exists
ERROR:  relation "film_actor" already exists
ERROR:  relation "film_category" already exists
ERROR:  relation "films_id_seq" already exists
ERROR:  relation "films" already exists
ERROR:  relation "genres_id_seq" already exists
ERROR:  relation "genres" already exists
ERROR:  relation "invoicelines_id_seq" already exists
ERROR:  relation "invoice_lines" already exists
E

CREATE TABLE


NOTICE:  relation "jtracks" already exists, skipping


COPY 318


In [115]:
import psycopg2

# Connect to the PostgreSQL database running in the Docker container
conn = psycopg2.connect(
    dbname="postgres",
    user="admin",
    password="password",
    host="localhost",
    port="5432"
)
cursor = conn.cursor()

# SQL query to order artists by listen_count
sql_query = """
WITH track_listen_count AS (
    SELECT
        artists.name AS artist,
        COUNT(*) AS listen_count
    FROM
        public.artists
    JOIN
        public.albums ON artists.id = albums.artist_id
    JOIN
        public.tracks ON albums.id = tracks.album_id
    JOIN
        public.invoice_lines ON tracks.id = invoice_lines.track_id
    GROUP BY
        artists.name
    UNION ALL
    SELECT
        artists.name AS artist,
        COUNT(*) AS listen_count
    FROM
        public.artists
    JOIN
        public.albums ON artists.id = albums.artist_id
    JOIN
        public.tracks ON albums.id = tracks.album_id
    JOIN
        public.jtracks ON tracks.name = jtracks.body->>'name'
    GROUP BY
        artists.name
)
SELECT
    artist,
    SUM(listen_count) AS total_listen_count
FROM
    track_listen_count
GROUP BY
    artist
ORDER BY
    total_listen_count DESC;
"""

# Execute the query
cursor.execute(sql_query)

# Fetch the results
results = cursor.fetchall()

# Close the database connection
conn.close()

# Print the results
for artist, total_listen_count in results:
    print(f"Artist: {artist}, Total Listen Count: {total_listen_count}")


Artist: Led Zeppelin, Total Listen Count: 152
Artist: Iron Maiden, Total Listen Count: 140
Artist: U2, Total Listen Count: 127
Artist: Metallica, Total Listen Count: 91
Artist: Queen, Total Listen Count: 87
Artist: Os Paralamas Do Sucesso, Total Listen Count: 45
Artist: Deep Purple, Total Listen Count: 44
Artist: Faith No More, Total Listen Count: 42
Artist: Lost, Total Listen Count: 41
Artist: Eric Clapton, Total Listen Count: 40
Artist: Ozzy Osbourne, Total Listen Count: 39
Artist: R.E.M., Total Listen Count: 39
Artist: Pearl Jam, Total Listen Count: 37
Artist: Creedence Clearwater Revival, Total Listen Count: 37
Artist: Guns N' Roses, Total Listen Count: 36
Artist: Titãs, Total Listen Count: 34
Artist: Green Day, Total Listen Count: 33
Artist: Kiss, Total Listen Count: 31
Artist: Van Halen, Total Listen Count: 29
Artist: Various artistss, Total Listen Count: 29
Artist: Red Hot Chili Peppers, Total Listen Count: 27
Artist: Chico Buarque, Total Listen Count: 27
Artist: Lenny Kravitz, 

### Generate an invoice report

The "counts" field represents the number a specific track has been listened to.
Based on the on the pricing information in the tracks table in, generate a report that calculates the total value per track.

One count equals one single order. 


Further Explanation: If the price of a single track is 0.99 and its count equals 10, the total equals 9.90.

In [118]:
import psycopg2

# Connect to the PostgreSQL database running in the Docker container
conn = psycopg2.connect(
    dbname="postgres",
    user="admin",
    password="password",
    host="localhost",
    port="5432"
)
cursor = conn.cursor()

# SQL query to calculate total value per track
sql_query = """
WITH track_listen_count AS (
    -- Query to get the total listen count per track
    SELECT
        tracks.id,
        COUNT(*) AS listen_count
    FROM
        public.tracks
    JOIN
        public.invoice_lines ON tracks.id = invoice_lines.track_id
    GROUP BY
        tracks.id
),
track_total_value AS (
    -- Query to calculate the total value per track
    SELECT
        tracks.id,
        tracks.name,
        tracks.unit_price,
        tlc.listen_count,
        tracks.unit_price * tlc.listen_count AS total_value
    FROM
        public.tracks
    JOIN
        track_listen_count tlc ON tracks.id = tlc.id
)
SELECT
    name AS track_name,
    unit_price,
    listen_count AS count,
    total_value
FROM
    track_total_value
ORDER BY
    total_value DESC;
"""

# Execute the query
cursor.execute(sql_query)

# Fetch the results
results = cursor.fetchall()

# Close the database connection
conn.close()

# Print the header
print("{:<50} {:<12} {:<7} {:<12}".format("Track Name", "Unit Price", "Count", "Total Value"))
print("-" * 85)

# Print the results
for track_name, unit_price, count, total_value in results:
    print("{:<50} ${:<12.2f} {:<7} ${:<12.2f}".format(track_name[:50], unit_price, count, total_value))


Track Name                                         Unit Price   Count   Total Value 
-------------------------------------------------------------------------------------
How to Stop an Exploding Man                       $1.99         2       $3.98        
The Woman King                                     $1.99         2       $3.98        
Walkabout                                          $1.99         2       $3.98        
The Fix                                            $1.99         2       $3.98        
Pilot                                              $1.99         2       $3.98        
Phyllis's Wedding                                  $1.99         2       $3.98        
Hot Girl                                           $1.99         2       $3.98        
Gay Witch Hunt                                     $1.99         2       $3.98        
Adrift                                             $1.99         1       $1.99        
The Glass Ballerina                           

### Most valued artists

Similar to the previous question but grouped by the corresponding artist: Using the some information in jtracks, group the total amount by the artist name.

Your result should look like:

| artist_name | total | 
|-------------|-------|
| AC/DC       | 312.23|
| Rammstein   | 200.21|
| Queens      |  33.12| 

In [119]:
import psycopg2

# Connect to the PostgreSQL database running in the Docker container
conn = psycopg2.connect(
    dbname="postgres",
    user="admin",
    password="password",
    host="localhost",
    port="5432"
)
cursor = conn.cursor()

# SQL query to calculate total income per artist
sql_query = """
WITH artist_total_income AS (
    SELECT
        public.jtracks.body->>'artist' AS artist_name,
        SUM((public.jtracks.body->>'count')::numeric * public.tracks.unit_price) AS total_income
    FROM
        public.jtracks
    JOIN
        public.tracks ON public.jtracks.body->>'name' = public.tracks.name
    GROUP BY
        public.jtracks.body->>'artist'
)
SELECT
    artist_name,
    total_income
FROM
    artist_total_income
ORDER BY
    total_income DESC;
"""

# Execute the query
cursor.execute(sql_query)

# Fetch the results
results = cursor.fetchall()

# Close the database connection
conn.close()

# Print the header
print("{:<90} {:<12}".format("artist_name", "total"))
print("-" * 100)

# Print the results
for artist_name, total_income in results:
    print("{:<90} {:<12.2f}".format(artist_name, total_income))


artist_name                                                                                total       
----------------------------------------------------------------------------------------------------
Led Zeppelin                                                                               6336.00     
Hollie Smith                                                                               3583.80     
David Osborne                                                                              2673.00     
La Esperanza                                                                               1856.25     
Queen                                                                                      1618.65     
Black Sabbath                                                                              549.45      
Bryan Lee                                                                                  470.25      
AC/DC                                                              

### Missing talent

There are some artists that are not in our music-store. Of those missing in our music-store, figure out the ones that should be signed us.

Query the top 5 artists that have the most total count and are not in our music store. Order your result descending.

In [120]:
import psycopg2

# Connect to the PostgreSQL database running in the Docker container
conn = psycopg2.connect(
    dbname="postgres",
    user="admin",
    password="password",
    host="localhost",
    port="5432"
)
cursor = conn.cursor()

# SQL query to find the top 5 missing artists with the most total count
sql_query = """
WITH missing_artists AS (
    -- Identify artists not present in our music store
    SELECT
        jtracks.body->>'artist' AS artist_name,
        COUNT(*) AS total_count
    FROM
        public.jtracks
    LEFT JOIN
        public.artists ON jtracks.body->>'artist' = artists.name
    WHERE
        artists.name IS NULL
    GROUP BY
        jtracks.body->>'artist'
    ORDER BY
        total_count DESC
    LIMIT 5
)
SELECT
    artist_name,
    total_count
FROM
    missing_artists
ORDER BY
    total_count DESC;
"""

# Execute the query
cursor.execute(sql_query)

# Fetch the results
results = cursor.fetchall()

# Close the database connection
conn.close()

# Print the header
print("{:<90} {:<12}".format("artist_name", "total_count"))
print("-" * 100)

# Print the results
for artist_name, total_count in results:
    print("{:<90} {:<12}".format(artist_name, total_count))


artist_name                                                                                total_count 
----------------------------------------------------------------------------------------------------
IEEE Computer Society                                                                      170         
Recording by Dr. Chuck                                                                     165         
Created by Sakai                                                                           80          
The Canettes Blues Band                                                                    80          
Kaiser Chiefs                                                                              75          
